In [1]:
import sys
print(sys.executable)

c:\Users\Matthew Montero\Downloads\Projects\mzmn-datathon-26\.env\Scripts\python.exe


In [50]:
# Load Dataset
import pandas as pd

housing_dataset = "data/Housing.csv"
df = pd.read_csv(housing_dataset)
print(df) # Prints out first 5 and last 5 tuples
print(df.columns) # Prints out all columns

        price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0    13300000  7420         4          2        3      yes        no       no   
1    12250000  8960         4          4        4      yes        no       no   
2    12250000  9960         3          2        2      yes        no      yes   
3    12215000  7500         4          2        2      yes        no      yes   
4    11410000  7420         4          1        2      yes       yes      yes   
..        ...   ...       ...        ...      ...      ...       ...      ...   
540   1820000  3000         2          1        1      yes        no      yes   
541   1767150  2400         3          1        1       no        no       no   
542   1750000  3620         2          1        1      yes        no       no   
543   1750000  2910         3          1        1       no        no       no   
544   1750000  3850         3          1        2      yes        no       no   

    hotwaterheating aircond

In [58]:
# Preprocessing Part 1
# Associate X and y values

X = df.drop(columns=["price"])
y = df["price"]

# print(X.shape)
# print(y.shape)

In [59]:
# Preprocessing Part 2
# Need to deal with categorical data

X_encoded = pd.get_dummies(X, drop_first=True) # One-hot encoding for categorical variables
# print(X_encoded.head())

In [84]:
# Train-Test Split 1: 70/30
# Train: Trains model
# Validation: Helps tune hyperparameters and checks for overfitting
# Test: Evaluates model performance. DO NOT TOUCH UNTIL END
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
random_state = 42

# Train and (Validation + Test) Split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=random_state)
# print("X_train: " + str(X_train.shape))
# print("X_test: " + str(X_test.shape))
# print("y_train: " + str(y_train.shape))
# print("y_test: " + str(y_test.shape))

# Transform features to have mean=0 and variance=1 - makes it easier for the model to learn
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# print("X_train_scaled: " + str(X_train_scaled))
# print("X_test_scaled: " + str(X_test_scaled))

In [85]:
# Model 1 - Base Model
from sklearn.linear_model import LinearRegression

base_model = LinearRegression()
base_model.fit(X_train_scaled, y_train)
y_base_pred = base_model.predict(X_test_scaled)

In [ ]:
# Model 1 Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

r2_1 = r2_score(y_test, y_base_pred) # higher is better, up to 1.0
# lower is better, down to 0.0: 
mse_1 = mean_squared_error(y_test, y_base_pred) # average of the squared differences between predicted and actual values
rmse_1 = root_mean_squared_error(y_test, y_base_pred) # same as MSE but easier to interpret than MSE
mae_1 = mean_absolute_error(y_test, y_base_pred) # average absolute difference between predicted and actual values
print(f"R²: {r2_1:.3f}, MSE: {mse_1:.3f}, RMSE: {rmse_1:.3f}, MAE: {mae_1:.3f}")

print("Coefficients:", base_model.coef_)
print("Intercept:", base_model.intercept_)

R²: 0.646, MSE: 1523019469501.291, RMSE: 1234106.750, MAE: 920392.940
Coefficients: [ 550762.64317168   60813.4371982   528860.12634351  359569.57150531
  253425.41739941  140091.96650564  107358.98663031  231590.48273824
  137463.98715129  319283.72367264  216284.0140147   -60189.21697981
 -180876.86582971]
Intercept: 4762406.272965879


In [ ]:
# Preprocessing Part 3
from sklearn.preprocessing import PolynomialFeatures

# Creates polynomial features (e.g., x1, x2, x1^2, x1*x2, x2^2 for degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)


In [ ]:
# Model 2 - Polynomial/Interaction Features

model_2 = LinearRegression()
model_2.fit(X_train_poly, y_train)
y_pred_2 = model_2.predict(X_test_poly)

In [ ]:
# Model 2 Evaluation

r2_2 = r2_score(y_test, y_base_pred) # higher is better, up to 1.0
# lower is better, down to 0.0: 
mse_2 = mean_squared_error(y_test, y_pred_2) # average of the squared differences between predicted and actual values
rmse_2 = root_mean_squared_error(y_test, y_pred_2) # same as MSE but easier to interpret than MSE
mae_2 = mean_absolute_error(y_test, y_pred_2) # average absolute difference between predicted and actual values
print(f"R²: {r2_2:.3f}, MSE: {mse_2:.3f}, RMSE: {rmse_2:.3f}, MAE: {mae_2:.3f}")

print("Coefficients:", base_model.coef_)
print("Intercept:", base_model.intercept_)

R²: 0.646, MSE: 1839557541903.196, RMSE: 1356302.895, MAE: 1000925.280
Coefficients: [ 550762.64317168   60813.4371982   528860.12634351  359569.57150531
  253425.41739941  140091.96650564  107358.98663031  231590.48273824
  137463.98715129  319283.72367264  216284.0140147   -60189.21697981
 -180876.86582971]
Intercept: 4762406.272965879


In [ ]:
# Model 3 - Ridge, Model 4 - Lasso
from sklearn.linear_model import Ridge, Lasso
# from sklearn.model_selection import GridSearchCV

